# 🏷️ Telecom VoC Comment Type (Operational Category) Intelligence Suite
### Multi-Class Machine Learning & Deep Learning Classification on Google Colab (GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rh61632/telecom-Voice-of-Customer_intelligence/blob/main/dl/telecom_voc_comment_type_classification_colab.ipynb)

---

### 📋 Overview & Problem Statement
In telecom Voice-of-Customer intelligence, analyzing **Sentiment** (Positive/Neutral/Negative) is only half the picture. To take operational action, telecom operators (Grameenphone, Robi, Banglalink) need to know the **Type / Category** of each comment:

1. **`General Appreciation / Other`**: General praise, emojis, UI comments.
2. **`Offers & Data Packs`**: Pricing complaints, MB pack requests, offer availability.
3. **`App Login & Technical Bugs`**: OTP issues, update errors, app crashes, biometric bugs.
4. **`Network Speed & 4G Latency`**: Slow internet, poor 4G coverage, buffering.
5. **`Billing & Airtime Deductions`**: Unexpected balance cuts, VAS deductions, recharge delays.

This notebook trains and benchmarks **4 distinct model paradigms** across 5-Fold Stratified Cross-Validation:
- 🏆 **Model 1: Soft-Voting Ensemble Machine Learning** (Character 3-5 n-grams + Word 1-2 n-grams TF-IDF)
- 🧠 **Model 2: Hybrid BiLSTM + Additive Bahdanau Attention + Char-CNN** (PyTorch GPU)
- 🚀 **Model 3: Multilingual Pretrained Transformer** (`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`)
- 🇧🇩 **Model 4: BUET BanglaBERT** (`csebuetnlp/banglabert`)


In [ ]:
# 1. Verify GPU Acceleration
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    device = torch.device("cuda")
else:
    print("WARNING: CUDA is not available. Training will proceed on CPU.")
    device = torch.device("cpu")


In [ ]:
# 2. Clone Repository & Setup Environment
import os
from pathlib import Path

if not Path("data/processed").exists():
    !git clone https://github.com/rh61632/telecom-Voice-of-Customer_intelligence.git
    %cd telecom-Voice-of-Customer_intelligence

!pip install -q transformers sentence-transformers scikit-learn pandas numpy matplotlib seaborn joblib


In [ ]:
# 3. Load & Preprocess Operational Category Dataset (N = 4,499)
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

LABEL_MAP = {
    "General Appreciation / Other": 0,
    "Offers & Data Packs": 1,
    "App Login & Technical Bugs": 2,
    "Network Speed & 4G Latency": 3,
    "Billing & Airtime Deductions": 4,
}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

files = {
    "Grameenphone": "data/processed/mygp_classified_reviews.csv",
    "Banglalink": "data/processed/mybl_classified_reviews.csv",
    "Robi": "data/processed/myrobi_classified_reviews.csv",
}

dfs = []
for op, p in files.items():
    d = pd.read_csv(p)
    if "operator" not in d.columns:
        d["operator"] = op
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)
df = df.dropna(subset=["review_text", "category"]).copy()
df["review_text"] = df["review_text"].astype(str).str.strip()
df = df[df["review_text"].str.len() > 0]
df = df[df["category"].isin(LABEL_MAP.keys())].reset_index(drop=True)
df["label"] = df["category"].map(LABEL_MAP)

# --- SHORT TEXT FILTERING TOGGLE ---
# Set MIN_WORDS = 1 to keep all reviews (N = 4,499)
# Set MIN_WORDS = 3 to exclude very small 1-2 word messages (N = 3,886)
# Set MIN_WORDS = 5 to keep only detailed feedback (N = 2,285)
MIN_WORDS = 1  # <-- Change this value to experiment with short-text filtering!
df["word_count"] = df["review_text"].str.split().apply(len)
if MIN_WORDS > 1:
    print(f"Applying filter: Keeping reviews with >= {MIN_WORDS} words...")
    df = df[df["word_count"] >= MIN_WORDS].reset_index(drop=True)
    print(f"Retained {len(df)} substantive reviews.")

def clean_text(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"(.){2,}", r"", text)
    text = re.sub(r"\s+", " ", text)
    return text.lower()

df["clean_text"] = df["review_text"].apply(clean_text)

print(f"Successfully loaded {len(df)} total category-labeled reviews across 3 operators.")
print("\nCategory Class Distribution:")
for cat, count in df["category"].value_counts().items():
    pct = count / len(df) * 100
    print(f"  - {cat:32s}: {count:5d} ({pct:5.2f}%)")

# Plot Class Distribution
plt.figure(figsize=(10, 4), dpi=120)
palette = ["#0090ff", "#ff7a00", "#e60000", "#9b59b6", "#2ecc71"]
sns.barplot(x=df["category"].value_counts().values, y=df["category"].value_counts().index, palette=palette)
plt.title("Distribution of Operational Comment Categories (N = 4,499)", fontsize=13, fontweight="bold")
plt.xlabel("Number of Reviews")
plt.tight_layout()
plt.show()



---

## 🏆 Model 1: Soft-Voting Ensemble Machine Learning
Combines three distinct inductive biases via soft probability averaging:
1. **Primary Balanced Logistic Regression**
2. **Challenger Calibrated LinearSVC** (Platt Scaling)
3. **Complement Naive Bayes** (Optimized for imbalanced class distributions)

Features are extracted using a **Dual Feature Union**:
- Character N-grams ($3-5$ chars) for phonetic Banglish spelling invariance
- Word N-grams ($1-2$ words) for domain keyword recognition (*"mb"*, *"offer"*, *"net"*, *"speed"*, *"login"*, *"taka"*).


In [ ]:
# 4. Train & Evaluate Soft-Voting Ensemble ML (5-Fold Stratified CV)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import classification_report, confusion_matrix

feat_union = FeatureUnion([
    ("word_tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("char_tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, sublinear_tf=True))
])

clf_lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
clf_svc = CalibratedClassifierCV(LinearSVC(class_weight="balanced", random_state=42))
clf_cnb = ComplementNB(norm=True)

ensemble_pipeline = Pipeline([
    ("features", feat_union),
    ("voting", VotingClassifier(
        estimators=[("lr", clf_lr), ("svc", clf_svc), ("cnb", clf_cnb)],
        voting="soft",
        weights=[2, 2, 1]
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Evaluating Soft-Voting Ensemble on 5-Fold Stratified CV...")
cv_scores = cross_validate(
    ensemble_pipeline,
    df["review_text"],
    df["category"],
    cv=cv,
    scoring={"macro_f1": "f1_macro", "weighted_f1": "f1_weighted", "accuracy": "accuracy"},
    n_jobs=-1
)

ensemble_macro_f1 = np.mean(cv_scores["test_macro_f1"])
ensemble_weighted_f1 = np.mean(cv_scores["test_weighted_f1"])
ensemble_acc = np.mean(cv_scores["test_accuracy"])

print(f"\nEnsemble ML Results (5-Fold CV):")
print(f"  Macro-F1    : {ensemble_macro_f1:.4f} (+/- {np.std(cv_scores['test_macro_f1']):.4f})")
print(f"  Weighted-F1 : {ensemble_weighted_f1:.4f}")
print(f"  Accuracy    : {ensemble_acc * 100:.2f}% (+/- {np.std(cv_scores['test_accuracy']) * 100:.2f}%)")


---

## 🧠 Model 2: Hybrid BiLSTM with Bahdanau Self-Attention & Char-CNN
- **Word Embeddings**: 128-dimensional dense representation
- **Character-CNN**: 64 filters over 3-character kernels to capture morphological sub-word structures
- **Bidirectional LSTM**: 2 recurrent layers capturing context in forward and reverse directions
- **Additive Self-Attention**: Learns dynamic attention distribution across the sequence
- **Loss Function**: `nn.CrossEntropyLoss` with inverse class-frequency weights to address class imbalance


In [ ]:
# 5. Architecture Definition: BiLSTM with Self-Attention & Char-CNN
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class Vocabulary:
    def __init__(self, max_words=10000, min_word_freq=2, max_chars=300):
        self.max_words = max_words
        self.min_word_freq = min_word_freq
        self.max_chars = max_chars
        self.pad_token, self.unk_token = "<pad>", "<unk>"
        self.word2idx = {self.pad_token: 0, self.unk_token: 1}
        self.char2idx = {self.pad_token: 0, self.unk_token: 1}

    def fit(self, texts):
        wc, cc = {}, {}
        for text in texts:
            for w in text.split(): wc[w] = wc.get(w, 0) + 1
            for c in text: cc[c] = cc.get(c, 0) + 1
        for w, cnt in sorted(wc.items(), key=lambda x: x[1], reverse=True)[:self.max_words]:
            if cnt >= self.min_word_freq and w not in self.word2idx:
                self.word2idx[w] = len(self.word2idx)
        for c in sorted(cc.keys(), key=lambda x: cc[x], reverse=True)[:self.max_chars]:
            if c not in self.char2idx:
                self.char2idx[c] = len(self.char2idx)

    def encode(self, text, max_words=64, max_word_chars=16):
        words = text.split()[:max_words]
        w_ids = [self.word2idx.get(w, 1) for w in words] + [0] * (max_words - len(words))
        c_matrix = []
        for w in words:
            c_ids = [self.char2idx.get(c, 1) for c in w[:max_word_chars]] + [0] * (max_word_chars - len(w[:max_word_chars]))
            c_matrix.append(c_ids)
        c_matrix += [[0] * max_word_chars for _ in range(max_words - len(words))]
        return w_ids, c_matrix

class TelecomReviewDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.labels = labels
        self.w, self.c = [], []
        for t in texts:
            w_ids, c_ids = vocab.encode(t)
            self.w.append(w_ids)
            self.c.append(c_ids)

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return (torch.tensor(self.w[idx], dtype=torch.long),
                torch.tensor(self.c[idx], dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long))

class BiLSTMWithAttention(nn.Module):
    def __init__(self, word_vocab_size, char_vocab_size, num_classes=5):
        super().__init__()
        self.word_emb = nn.Embedding(word_vocab_size, 128, padding_idx=0)
        self.char_emb = nn.Embedding(char_vocab_size, 32, padding_idx=0)
        self.char_conv = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(0.3)
        self.bilstm = nn.LSTM(128 + 64, 128, num_layers=2, bidirectional=True, batch_first=True, dropout=0.3)
        self.proj = nn.Linear(256, 128)
        self.query = nn.Linear(128, 1, bias=False)
        self.fc = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, w_ids, c_ids):
        b, s = w_ids.size()
        w = self.word_emb(w_ids)
        c = self.char_emb(c_ids.view(b * s, -1)).transpose(1, 2)
        c_conv = F.relu(self.char_conv(c))
        c_feat = torch.max(c_conv, dim=-1)[0].view(b, s, -1)
        x = self.dropout(torch.cat([w, c_feat], dim=-1))
        lstm_out, _ = self.bilstm(x)
        
        # Attention
        energy = torch.tanh(self.proj(lstm_out))
        scores = self.query(energy).squeeze(-1).masked_fill(w_ids == 0, -1e9)
        weights = F.softmax(scores, dim=-1).unsqueeze(1)
        context = torch.bmm(weights, lstm_out).squeeze(1)
        return self.fc(context)


In [ ]:
# 6. Train & Evaluate BiLSTM on GPU (5-Fold Stratified CV)
from sklearn.metrics import f1_score, accuracy_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
bilstm_f1_scores = []
bilstm_acc_scores = []

class_counts = df["label"].value_counts().sort_index().values
weights = torch.tensor([len(df) / (len(class_counts) * c) for c in class_counts], dtype=torch.float32).to(device)

print(f"Training BiLSTM + Attention across 5 Folds on {device}...")
for fold, (tr_idx, val_idx) in enumerate(skf.split(df, df["label"]), 1):
    tr_df, val_df = df.iloc[tr_idx], df.iloc[val_idx]
    vocab = Vocabulary()
    vocab.fit(tr_df["clean_text"].tolist())
    
    tr_loader = DataLoader(TelecomReviewDataset(tr_df["clean_text"].tolist(), tr_df["label"].tolist(), vocab), batch_size=64, shuffle=True)
    val_loader = DataLoader(TelecomReviewDataset(val_df["clean_text"].tolist(), val_df["label"].tolist(), vocab), batch_size=128, shuffle=False)
    
    model = BiLSTMWithAttention(len(vocab.word2idx)+1, len(vocab.char2idx)+1, num_classes=5).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(weight=weights)
    
    best_f1 = 0.0
    for epoch in range(1, 9):
        model.train()
        for w_b, c_b, y_b in tr_loader:
            w_b, c_b, y_b = w_b.to(device), c_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(w_b, c_b), y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
        
        # Validation
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for w_b, c_b, y_b in val_loader:
                w_b, c_b = w_b.to(device), c_b.to(device)
                logits = model(w_b, c_b)
                preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
                targets.extend(y_b.numpy())
        f1 = f1_score(targets, preds, average="macro", zero_division=0)
        if f1 > best_f1: best_f1 = f1
        
    acc = accuracy_score(targets, preds)
    bilstm_f1_scores.append(best_f1)
    bilstm_acc_scores.append(acc)
    print(f"  Fold {fold}/5: Best Macro-F1 = {best_f1:.4f} | Accuracy = {acc*100:.2f}%")

bilstm_macro_f1 = np.mean(bilstm_f1_scores)
bilstm_acc = np.mean(bilstm_acc_scores)
print(f"\nBiLSTM Mean Macro-F1: {bilstm_macro_f1:.4f} (+/- {np.std(bilstm_f1_scores):.4f}) | Mean Accuracy: {bilstm_acc*100:.2f}%")


---

## 🚀 Model 3: Multilingual Pretrained Transformer (MiniLM)
Uses `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` to extract 384-dimensional dense semantic sentence representations with GPU acceleration, followed by an MLP classification head.


In [ ]:
# 7. Extract Multilingual Transformer Embeddings on GPU
from transformers import AutoTokenizer, AutoModel

MINILM_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print(f"Loading {MINILM_ID} onto {device}...")
tokenizer_minilm = AutoTokenizer.from_pretrained(MINILM_ID)
model_minilm = AutoModel.from_pretrained(MINILM_ID).to(device)
model_minilm.eval()

def extract_embeddings(texts, tok, mod, bsize=128):
    embs = []
    for i in range(0, len(texts), bsize):
        batch = texts[i:i+bsize]
        encoded = tok(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            out = mod(**encoded)
            mask = encoded["attention_mask"].unsqueeze(-1)
            # Mean pooling
            p = (out.last_hidden_state * mask).sum(dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)
            embs.append(p.cpu().numpy())
    return np.vstack(embs)

print("Extracting MiniLM embeddings for all 4,499 reviews on GPU...")
minilm_embeddings = extract_embeddings(df["review_text"].tolist(), tokenizer_minilm, model_minilm)
print(f"Extracted shape: {minilm_embeddings.shape}")


In [ ]:
# 8. Train & Evaluate Multilingual Transformer Head (5-Fold CV)
class TransformerHead(nn.Module):
    def __init__(self, in_features=384, num_classes=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.net(x)

minilm_f1_scores = []
minilm_acc_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(df, df["label"]), 1):
    X_tr, y_tr = torch.tensor(minilm_embeddings[tr_idx], dtype=torch.float32), torch.tensor(df["label"].iloc[tr_idx].values, dtype=torch.long)
    X_val, y_val = torch.tensor(minilm_embeddings[val_idx], dtype=torch.float32), torch.tensor(df["label"].iloc[val_idx].values, dtype=torch.long)
    
    loader_tr = DataLoader(torch.utils.data.TensorDataset(X_tr, y_tr), batch_size=128, shuffle=True)
    loader_val = DataLoader(torch.utils.data.TensorDataset(X_val, y_val), batch_size=256, shuffle=False)
    
    head = TransformerHead(in_features=384, num_classes=5).to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(weight=weights)
    
    best_f1 = 0.0
    for epoch in range(1, 15):
        head.train()
        for x_b, y_b in loader_tr:
            x_b, y_b = x_b.to(device), y_b.to(device)
            opt.zero_grad()
            loss = crit(head(x_b), y_b)
            loss.backward()
            opt.step()
            
        head.eval()
        preds, targets = [], []
        with torch.no_grad():
            for x_b, y_b in loader_val:
                x_b = x_b.to(device)
                preds.extend(torch.argmax(head(x_b), dim=-1).cpu().numpy())
                targets.extend(y_b.numpy())
        f1 = f1_score(targets, preds, average="macro", zero_division=0)
        if f1 > best_f1: best_f1 = f1
        
    acc = accuracy_score(targets, preds)
    minilm_f1_scores.append(best_f1)
    minilm_acc_scores.append(acc)
    print(f"  Fold {fold}/5: MiniLM Best Macro-F1 = {best_f1:.4f} | Accuracy = {acc*100:.2f}%")

minilm_macro_f1 = np.mean(minilm_f1_scores)
minilm_acc = np.mean(minilm_acc_scores)
print(f"\nMiniLM Transformer Mean Macro-F1: {minilm_macro_f1:.4f} (+/- {np.std(minilm_f1_scores):.4f}) | Mean Accuracy: {minilm_acc*100:.2f}%")


---

## 🇧🇩 Model 4: BUET BanglaBERT (`csebuetnlp/banglabert`)
Pretrained on 110 GB of native Bengali corpora by BUET NLP. Provides strong cross-lingual representations for Bangla script and telecom domain vocabulary.


In [ ]:
# 9. Extract BUET BanglaBERT Embeddings on GPU
BANGLABERT_ID = "csebuetnlp/banglabert"
print(f"Loading {BANGLABERT_ID} onto {device}...")
tokenizer_bbert = AutoTokenizer.from_pretrained(BANGLABERT_ID)
model_bbert = AutoModel.from_pretrained(BANGLABERT_ID).to(device)
model_bbert.eval()

print("Extracting BUET BanglaBERT embeddings (768-dim) on GPU...")
bbert_embeddings = extract_embeddings(df["review_text"].tolist(), tokenizer_bbert, model_bbert)
print(f"Extracted shape: {bbert_embeddings.shape}")


In [ ]:
# 10. Train & Evaluate BanglaBERT Head (5-Fold CV)
bbert_f1_scores = []
bbert_acc_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(df, df["label"]), 1):
    X_tr, y_tr = torch.tensor(bbert_embeddings[tr_idx], dtype=torch.float32), torch.tensor(df["label"].iloc[tr_idx].values, dtype=torch.long)
    X_val, y_val = torch.tensor(bbert_embeddings[val_idx], dtype=torch.float32), torch.tensor(df["label"].iloc[val_idx].values, dtype=torch.long)
    
    loader_tr = DataLoader(torch.utils.data.TensorDataset(X_tr, y_tr), batch_size=128, shuffle=True)
    loader_val = DataLoader(torch.utils.data.TensorDataset(X_val, y_val), batch_size=256, shuffle=False)
    
    head = TransformerHead(in_features=768, num_classes=5).to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(weight=weights)
    
    best_f1 = 0.0
    for epoch in range(1, 15):
        head.train()
        for x_b, y_b in loader_tr:
            x_b, y_b = x_b.to(device), y_b.to(device)
            opt.zero_grad()
            loss = crit(head(x_b), y_b)
            loss.backward()
            opt.step()
            
        head.eval()
        preds, targets = [], []
        with torch.no_grad():
            for x_b, y_b in loader_val:
                x_b = x_b.to(device)
                preds.extend(torch.argmax(head(x_b), dim=-1).cpu().numpy())
                targets.extend(y_b.numpy())
        f1 = f1_score(targets, preds, average="macro", zero_division=0)
        if f1 > best_f1: best_f1 = f1
        
    acc = accuracy_score(targets, preds)
    bbert_f1_scores.append(best_f1)
    bbert_acc_scores.append(acc)
    print(f"  Fold {fold}/5: BanglaBERT Best Macro-F1 = {best_f1:.4f} | Accuracy = {acc*100:.2f}%")

bbert_macro_f1 = np.mean(bbert_f1_scores)
bbert_acc = np.mean(bbert_acc_scores)
print(f"\nBanglaBERT Mean Macro-F1: {bbert_macro_f1:.4f} (+/- {np.std(bbert_f1_scores):.4f}) | Mean Accuracy: {bbert_acc*100:.2f}%")


---

## 📊 Model Comparison & Visualizations
Comprehensive benchmark comparing all 4 paradigm candidates on the 5-Fold Stratified Cross-Validation benchmark.


In [ ]:
# 11. Visual Benchmark Summary across All Paradigms
summary_df = pd.DataFrame([
    {"Model Paradigm": "Soft-Voting Ensemble ML", "Macro F1": ensemble_macro_f1, "Accuracy": ensemble_acc},
    {"Model Paradigm": "BiLSTM + Attention (GPU)", "Macro F1": bilstm_macro_f1, "Accuracy": bilstm_acc},
    {"Model Paradigm": "Multilingual MiniLM Head", "Macro F1": minilm_macro_f1, "Accuracy": minilm_acc},
    {"Model Paradigm": "BUET BanglaBERT Head", "Macro F1": bbert_macro_f1, "Accuracy": bbert_acc},
]).sort_values("Macro F1", ascending=False).reset_index(drop=True)

print("=" * 60)
print("OPERATIONAL COMMENT TYPE BENCHMARK SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

# Plot Comparison
fig, ax1 = plt.subplots(figsize=(10, 4), dpi=120)
palette = ["#0090ff", "#2ecc71", "#e60000", "#9b59b6"]
bars = sns.barplot(data=summary_df, x="Model Paradigm", y="Macro F1", palette=palette, ax=ax1)
plt.title("Operational Category Classification: 5-Fold CV Macro-F1", fontsize=13, fontweight="bold")
plt.ylim(0, 1.0)
for bar in bars.patches:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f"{yval:.4f}", ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.show()


---

## 🧪 Interactive Real-Time Prediction
Test custom reviews in **Bangla**, **Banglish**, or **English** across all 5 operational categories.


In [ ]:
# 12. Train Final Production Models and Test Live Samples
print("Fitting final Soft-Voting Ensemble on full 4,499 dataset...")
ensemble_pipeline.fit(df["review_text"], df["category"])

test_queries = [
    "Network speed khub e baje, 4G thakleo net chole na",
    "Taka kete niyeche kintu MB dey nai, faltu system",
    "App open hocche na shudhu loading dekhasse otp ashe na",
    "Offer gulo onek dam, kom dame bhalo data pack chai",
    "Onek valo app, sob kichu sundor bhabe kora jay",
    "Customer care e kotha bola jay na, balance katse",
]

print("\n" + "=" * 70)
print("LIVE SAMPLE INFERENCE (SOFT-VOTING ENSEMBLE)")
print("=" * 70)
for q in test_queries:
    pred = ensemble_pipeline.predict([q])[0]
    probs = ensemble_pipeline.predict_proba([q])[0]
    conf = np.max(probs) * 100
    print(f"\nQuery    : \"{q}\"")
    print(f"Predicted: {pred} (Confidence: {conf:.1f}%)")


---

## 💾 Save & Download Checkpoints
Export trained category model weights and download them to your local machine.


In [ ]:
# 13. Save Model Checkpoints & Export results
import joblib

out_dir = Path("exported_category_models")
out_dir.mkdir(exist_ok=True)

model_path = out_dir / "category_voting_classifier.joblib"
joblib.dump(ensemble_pipeline, model_path)
print(f"Ensemble model saved to: {model_path}")

summary_df.to_csv(out_dir / "category_benchmark_summary.csv", index=False)
print("Saved category benchmark summary to CSV.")

try:
    from google.colab import files
    !zip -r category_models.zip exported_category_models/
    files.download("category_models.zip")
    print("Download triggered automatically.")
except Exception as e:
    print("Running in non-Colab environment or download skipped.")
